# Whisper-small fine-tune (Colab wrapper)

This notebook **only wraps** the `pasak` package. There is no second training loop here.

**Do not zip FLEURS from the PC into Colab.** `prepare_asr --fleurs` pulls `google/fleurs` `id_id` on this VM.

Local collect (not Colab): uvicorn host `127.0.0.1:8767`, static origin `http://127.0.0.1:8765` (repo root, not `localhost`). CORS fails otherwise.

Upload the `backend/` tree (or `src/` + `pyproject.toml`) and, optionally, a Common Voice Indonesian dump and `golden-v0.zip` from `python -m pasak.pack_golden`.

Checkpoint leaves the VM via **R2** (not `files.download`). Set Colab Secrets before the last code cell (see that markdown).

In [ ]:
# Point at the uploaded backend tree, then install with the ml extra.
# Example after uploading backend.zip to /content:
# !unzip -q backend.zip -d /content/backend
%pip install -e "/content/backend[ml]"

In [ ]:
# FLEURS on this VM. Optional Common Voice dump: add --cv-dir /content/cv
!python -m pasak.prepare_asr --fleurs

Optional expense golden (amount labels only). Unzip `golden-v0.zip` if you uploaded it. Do not pack FLEURS into that zip.

In [ ]:
# !unzip -q /content/golden-v0.zip -d /content/golden-v0

In [ ]:
!python -m pasak.finetune_whisper --asr /content/backend/data/asr/manifest.jsonl --out /content/whisper-small-ft
# optional: --golden /content/golden-v0/examples.jsonl

## R2 secrets (`userdata`)

`google.colab.userdata` reads **Colab Secrets**, not notebook cells and not `.env`.

1. Open this notebook on [colab.research.google.com](https://colab.research.google.com).
2. Left sidebar: the **key icon** (Secrets). If the bar is collapsed, hover the left edge.
3. **Add secret** four times (name must match exactly):
   - `R2_ACCOUNT_ID` — Cloudflare account id (endpoint host is `{id}.r2.cloudflarestorage.com`)
   - `R2_BUCKET` — private bucket (not the public Phase 2 `models` product bucket)
   - `R2_ACCESS_KEY_ID`
   - `R2_SECRET_ACCESS_KEY`
4. For **this notebook**, switch **Notebook access** ON for each secret (otherwise `userdata.get` raises).
5. Re-run the next cell. Secrets persist in your Google account; they are not in git.

Prefix on R2: `maintainer/whisper-small-ft/`. Skip `checkpoint-*` (optimizer). Do not paste keys into cells.

In [ ]:
%pip install -q boto3

from pathlib import Path

import boto3
from botocore.config import Config
from google.colab import userdata

account = userdata.get("R2_ACCOUNT_ID")
bucket = userdata.get("R2_BUCKET")
prefix = "maintainer/whisper-small-ft/"

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{account}.r2.cloudflarestorage.com",
    aws_access_key_id=userdata.get("R2_ACCESS_KEY_ID"),
    aws_secret_access_key=userdata.get("R2_SECRET_ACCESS_KEY"),
    region_name="auto",
    config=Config(s3={"addressing_style": "path"}),
)

root = Path("/content/whisper-small-ft")
for path in root.rglob("*"):
    if not path.is_file():
        continue
    if any(part.startswith("checkpoint-") for part in path.relative_to(root).parts):
        continue
    key = prefix + path.relative_to(root).as_posix()
    print("put", key, path.stat().st_size)
    s3.upload_file(str(path), bucket, key)
print("done")

# Fallback if R2 is not set (often fails on a large zip):
# !zip -r /content/whisper-small-ft.zip /content/whisper-small-ft -x "*/checkpoint-*/*"
# from google.colab import files
# files.download("/content/whisper-small-ft.zip")